#ENSEMBLE SOFT VOTING

In [5]:
import pandas as pd
import numpy as np
import math
from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (BertTokenizer, DistilBertTokenizer,
                          BertForSequenceClassification, DistilBertForSequenceClassification,
                          Trainer, TrainingArguments)

bert_lr = 0.00003
bert_bs = 32
bert_wd = 0.1


distil_lr = 0.00003
distil_bs = 16
distil_wd = 0.01


bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
distil_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


def run_ensemble_pipeline(file_path, dataset_name):
    print(f"\n{'='*60}")
    print(f"  STARTING PIPELINE FOR: {dataset_name}")
    print(f"{'='*60}\n")

    df = pd.read_excel(file_path)[['text', 'label']]
    labels_list = df['label'].unique().tolist()
    label2id = {label: i for i, label in enumerate(labels_list)}
    df['label'] = df['label'].map(label2id)
    num_labels = len(labels_list)

    dataset_split = Dataset.from_pandas(df).train_test_split(test_size=0.2, seed=42)
    train_dataset = dataset_split['train']
    eval_dataset = dataset_split['test']


    print(">>> Tokenizing data...")
    tokenized_train_bert = train_dataset.map(lambda e: bert_tokenizer(e['text'], padding='max_length', truncation=True, max_length=128), batched=True)
    tokenized_eval_bert = eval_dataset.map(lambda e: bert_tokenizer(e['text'], padding='max_length', truncation=True, max_length=128), batched=True)

    tokenized_train_distil = train_dataset.map(lambda e: distil_tokenizer(e['text'], padding='max_length', truncation=True, max_length=128), batched=True)
    tokenized_eval_distil = eval_dataset.map(lambda e: distil_tokenizer(e['text'], padding='max_length', truncation=True, max_length=128), batched=True)


    print(f"\n>>> Training BERT on {dataset_name}...")
    bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

    training_args_bert = TrainingArguments(
        output_dir=f'./temp_bert_{dataset_name}',
        num_train_epochs=1,
        learning_rate=bert_lr,
        per_device_train_batch_size=bert_bs,
        per_device_eval_batch_size=bert_bs,
        weight_decay=bert_wd,
        eval_strategy="no",
        save_strategy="no",
        report_to="none"
    )

    trainer_bert = Trainer(
        model=bert_model,
        args=training_args_bert,
        train_dataset=tokenized_train_bert,
        eval_dataset=tokenized_eval_bert
    )
    trainer_bert.train()

    print(f">>> Generating BERT Predictions for {dataset_name}...")
    bert_preds = trainer_bert.predict(tokenized_eval_bert)
    bert_probs = softmax(bert_preds.predictions, axis=1)


    print(f"\n>>> Training DistilBERT on {dataset_name}...")
    distil_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels)

    training_args_distil = TrainingArguments(
        output_dir=f'./temp_distil_{dataset_name}',
        num_train_epochs=1,
        learning_rate=distil_lr,
        per_device_train_batch_size=distil_bs,
        per_device_eval_batch_size=distil_bs,
        weight_decay=distil_wd,
        eval_strategy="no",
        save_strategy="no",
        report_to="none"
    )

    trainer_distil = Trainer(
        model=distil_model,
        args=training_args_distil,
        train_dataset=tokenized_train_distil,
        eval_dataset=tokenized_eval_distil
    )
    trainer_distil.train()

    print(f">>> Generating DistilBERT Predictions for {dataset_name}...")
    distil_preds = trainer_distil.predict(tokenized_eval_distil)
    distil_probs = softmax(distil_preds.predictions, axis=1)

    print(f"\n>>> Ensembling (Soft Voting) for {dataset_name}...")
    ensemble_probs = (bert_probs + distil_probs) / 2
    ensemble_final_preds = np.argmax(ensemble_probs, axis=1)

    actual_labels = tokenized_eval_bert['label']
    precision, recall, f1, _ = precision_recall_fscore_support(actual_labels, ensemble_final_preds, average='weighted', zero_division=0)
    acc = accuracy_score(actual_labels, ensemble_final_preds)

    print(f"\n{'='*50}")
    print(f"FINAL ENSEMBLE RESULTS: {dataset_name}")
    print(f"{'='*50}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"{'='*50}\n")




run_ensemble_pipeline(file_path="/content/clean_dataset_1_agnews.xlsx", dataset_name="dataset_1_AGNews")

run_ensemble_pipeline(file_path="/content/clean_dataset_2_legaldispute.xlsx", dataset_name="dataset_2_Legal")


  STARTING PIPELINE FOR: dataset_1_AGNews

>>> Tokenizing data...


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


>>> Generating DistilBERT Predictions for dataset_2_Legal...



>>> Ensembling (Soft Voting) for dataset_2_Legal...

FINAL ENSEMBLE RESULTS: dataset_2_Legal
Accuracy : 0.5100
Precision: 0.3799
Recall   : 0.5100
F1 Score : 0.3970

